In [ ]:
   !pip install --force-reinstall "pulp==3.3.2"

  Using cached pulp-3.3.2-py3-none-any.whl.metadata (8.6 kB)
Using cached pulp-3.3.2-py3-none-any.whl (16.4 MB)
  Attempting uninstall: pulp
    Found existing installation: PuLP 3.3.2
    Uninstalling PuLP-3.3.2:
      Successfully uninstalled PuLP-3.3.2


In [ ]:
   import pulp
   print(pulp.__version__)

3.3.2


In [ ]:
supply = {"Ikeja": 300, "Apapa": 250}
demand = {"Yaba": 90, "Surulere": 110, "Lekki": 130, "Ikorodu": 80, "Ajah": 100}

cost = {
    ("Ikeja", "Yaba"): 120,
    ("Ikeja", "Surulere"): 150,
    ("Ikeja", "Lekki"): 260,
    ("Ikeja", "Ikorodu"): 200,
    ("Ikeja", "Ajah"): 300,
    ("Apapa", "Yaba"): 140,
    ("Apapa", "Surulere"): 110,
    ("Apapa", "Lekki"): 210,
    ("Apapa", "Ikorodu"): 280,
    ("Apapa", "Ajah"): 230,
}

In [ ]:
print(cost[("Apapa", "Lekki")])

210


In [ ]:
print(cost[("Apapa", "Lekki")])    # should print 210
print(len(cost))                   # should print 10 (2 warehouses x 5 pharmacies)
print(sum(supply.values()), sum(demand.values()))   # should print 550 510

210
10
550 510


In [ ]:
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD

model = LpProblem("Pharmacy_Distribution", LpMinimize)

In [ ]:
ship = LpVariable.dicts("ship", cost.keys(), lowBound=0)

In [ ]:
model += lpSum(cost[route] * ship[route] for route in cost)

In [ ]:
for w in supply:
    model += lpSum(ship[(w, p)] for p in demand) <= supply[w]

for p in demand:
    model += lpSum(ship[(w, p)] for w in supply) >= demand[p]

In [ ]:
model.solve(PULP_CBC_CMD(msg=False))

1

In [ ]:
print("Status:", LpStatus[model.status])
print("Total cost: N", value(model.objective))
for route in cost:
    if ship[route].value() > 0:
        print(route[0], "->", route[1], ":", ship[route].value(), "cartons")

Status: Optimal
Total cost: N 92800.0
Ikeja -> Yaba : 90.0 cartons
Ikeja -> Surulere : 90.0 cartons
Ikeja -> Ikorodu : 80.0 cartons
Apapa -> Surulere : 20.0 cartons
Apapa -> Lekki : 130.0 cartons
Apapa -> Ajah : 100.0 cartons
